In [1]:
from mlds import data_loader
from mlds.utils.converter import simple_majority_vote
import pandas as pd

from pprint import pprint

entity_manager = data_loader.EntityDataManager(data_folder="data/json")
intent_manager = data_loader.IntentDataManager(
    data_folder="data/UtteranceGen/submission"
)

In [2]:
def to_entity_spans(data):
    result = []
    for annotation in data['annotations']:
        if annotation['completed_by']['id'] == 'MAJORITY_VOTE':
            for item in annotation['result']:
                start = item['value']['start']
                end = item['value']['end']
                label = item['value']['labels'][0]
                result.append(f"{start}:{end}:SL:{label}")

    return ','.join(sorted(result, key=lambda x: int(x.split(':')[0])))

def merge_entity_spans(text_a, spana, text_b, spanb, joint_char=""):
    # remember to add a space between the two texts
    # the start and end index of the second text should be updated
    l = []
    for item in spanb.split(","):
        if item == "":
            continue
        start, end, _, label = item.split(":")
        l.append( f"{int(start) + len(text_a) + len(joint_char)}:{int(end) + len(text_a) + len(joint_char)}:SL:{label}")
    spanb = ",".join(l)
    if spana:
        spana += "," + spanb
    return spanb


from pprint import  pprint

def to_logical_form(text, intent, spans):
    # [IN:GET_MESSAGE [SL:CONTACT Angelika Kratzer ] [SL:TYPE_CONTENT video ] [SL:RECIPIENT me ] ]
    intent = intent.replace(" ", "_")
    ss = spans.split(",")
    # if ss == [""]:
    #     return ""
    result = f"[IN:{intent} "
    for item in ss:
        if item == "":
            continue
        start, end, _, label = item.split(":")
        result += f"[SL:{label} {text[int(start):int(end)]}] "
    result += "]"

    return result


def merge_entity_intent(lan, lan_entity):
    print(f"[[[ Start {lan}")
    lan_entity = simple_majority_vote(lan_entity)
    # merge {'Muyinza okummanyisa ddi lwe muliteeka envumbo ku akawunti ya kkampuni',
    # 'Nsaba kumanya ebyetaagisa okuteeka envumbo ku akawunti yange eri mu',
    # 'Waliwo eyatadde ensimbi ku akawunti yange eyateereddwako envumbo mu',
    # 'bbanka ya Equity?',
    # 'bbanka ya Finance trust.',
    # 'yange mu ABC capital bbanka?'}
    # {'Muyinza okummanyisa ddi lwe muliteeka envumbo ku akawunti ya kkampuni\n'
    # 'yange mu ABC capital bbanka?',
    # 'Nsaba kumanya ebyetaagisa okuteeka envumbo ku akawunti yange eri mu\n'
    # 'bbanka ya Finance trust.',
    # 'Waliwo eyatadde ensimbi ku akawunti yange eyateereddwako envumbo mu\n'
    # 'bbanka ya Equity?'}
    
    lan_entity = pd.DataFrame([{
            "text": item["data"]["text"],
            "text4match": item["data"]["text"].replace('"', ''),
            "spans": to_entity_spans(item),
            # "changed": item["changed"],
        } for item in lan_entity])

    if lan == "lug":
        for a, b in [
            ("\"Muyinza okummanyisa ddi lwe muliteeka envumbo ku akawunti ya kkampuni",
                "yange mu ABC capital bbanka?\""),
            ("\"Nsaba kumanya ebyetaagisa okuteeka envumbo ku akawunti yange eri mu",
                "bbanka ya Finance trust.\""),
            ("\"Waliwo eyatadde ensimbi ku akawunti yange eyateereddwako envumbo mu",
                "bbanka ya Equity?\"")
        ]:
            a_entity = lan_entity[lan_entity["text"] == a]
            b_entity = lan_entity[lan_entity["text"] == b]
            
            lan_entity = lan_entity[lan_entity["text"] != a]
            lan_entity = lan_entity[lan_entity["text"] != b]
            
            lan_entity = pd.concat([lan_entity, pd.DataFrame({
                "text": [a + "\n" + b],
                "text4match": [(a + "\n" + b).replace('"', '')],
                "spans": [merge_entity_spans(a, a_entity["spans"].values[0], b, b_entity["spans"].values[0])],
            })], ignore_index=True)

    elif lan == "ibo":
        {'Biko Tinye egwu Only a Woman nke Enrique Iglesias na ndepụta  \n'
         'ndepụta ọkpụkpọ egwu ụra m.',
         'Biko tinyere m egwu Westlife na \nndepụta ọkpụkpọ egwu ịhụnanya m',
         'E nwere m egwu Bob Marley na \nndepụta ọkpụkpọ egwu m ọbụla?',
         'Tinye egwu a na-akpọ ugbua na \nndepụta ọkpụkpọ m na-ege mgbe m na-esi nri.',
         "Tinyere egwu I won't give up on us nke Jason Mraz na \n"
         'ndepụta ọkpụkpọ egwu ịhụnanya m.',
         'Tinyere m Egwu Count on me nke Bruno Mars na \n'
         'ndepụta ọkpụkpọ egwu ezumiike m.'}
        for a, b in [
            # \"Biko Tinye egwu \"\"Only a Woman\"\" nke Enrique Iglesias na ndepụta  
            ("\"Biko Tinye egwu \"\"Only a Woman\"\" nke Enrique Iglesias na ndepụta  ", "ndepụta ọkpụkpọ egwu ụra m.\""),
            ("\"Biko tinyere m egwu Westlife na ", "ndepụta ọkpụkpọ egwu ịhụnanya m\""),
            ("\"E nwere m egwu Bob Marley na ", "ndepụta ọkpụkpọ egwu m ọbụla?\""),
            ("\"Tinye egwu a na-akpọ ugbua na ", "ndepụta ọkpụkpọ m na-ege mgbe m na-esi nri.\""),
            ("\"Tinyere egwu \"\"I won't give up on us\"\" nke Jason Mraz na ", "ndepụta ọkpụkpọ egwu ịhụnanya m.\""),
            ("\"Tinyere m Egwu \"\"Count on me\"\" nke Bruno Mars na ", "ndepụta ọkpụkpọ egwu ezumiike m.\"")
        ]:
            a_entity = lan_entity[lan_entity["text"] == a]
            b_entity = lan_entity[lan_entity["text"] == b]
            
            lan_entity = lan_entity[lan_entity["text"] != a]
            lan_entity = lan_entity[lan_entity["text"] != b]
            
            lan_entity = pd.concat([lan_entity, pd.DataFrame({
                "text": [a + "\n" + b],
                "text4match": [(a + "\n" + b).replace('"', '')],
                "spans": [merge_entity_spans(a, a_entity["spans"].values[0], b, b_entity["spans"].values[0])],
            })], ignore_index=True)

    lan_intent = intent_manager.load_data(lan)
    if lan == "wol":
        lan_intent["text4match"] = lan_intent["text"].apply(lambda x: x.replace('"', '').strip())
    else:
        lan_intent["text4match"] = lan_intent["text"].apply(lambda x: x.replace('"', '').strip("\n"))

    lan_intent.drop(columns=["text"], inplace=True)

    # lan_entity = lan_entity[lan_entity["changed"]]
    print(f"lan_entity: {len(lan_entity)}")
    print(f"lan_intent: {len(lan_intent)}")

    # combine the two dataframes
    df = pd.merge(lan_intent, lan_entity, on="text4match")
    df["logical_form"]  = df.apply(lambda x: to_logical_form(x["text"], x["intent"], x["spans"]), axis=1)
    
    if lan == "lug":
        df.drop_duplicates(inplace=True)

    # df = df[df["changed"] == True]

    if len(df) != len(lan_intent) or len(df) != len(lan_entity):
        from collections import Counter

        entity_set = set(lan_entity["text4match"].tolist())
        intent_set = set(lan_intent["text4match"].tolist())
        df_set = set(df["text4match"].tolist())

        c = Counter(lan_entity["text4match"].tolist())
        if len([k for k, v in c.items() if v > 1]) > 0:
            l = ([(k,v) for k, v in c.items() if v > 1])
            ll = [item[0] for item in l]
            print(lan_entity[lan_entity["text4match"].isin(ll)])
            
        print(len(entity_set), len(intent_set), len(df_set))
        pprint(entity_set - intent_set)
        pprint(intent_set - entity_set)

        # print(df_set - entity_set)
        # print(df_set - intent_set)
    df.drop(columns=["text4match", "split"], inplace=True)
    df.to_csv(f"data/output/{lan}.csv", index=False)
    print(len(df))
    print(f"{lan} done, save file to data/output/{lan}.csv")

    if not (len(df) == len(lan_intent) == len(lan_entity) == 3200):
        print(f"Error: {lan} has {len(df)} rows, {len(lan_intent)} intents, {len(lan_entity)} entities")

    def sanity_check(data: pd.DataFrame, n: int = 80):
        # total have 3200 samples
        # 40 intent * 80 samples
        print(len(data))
        print(len(data["intent"].unique()))
        print(all(data.groupby("intent").size() == n))

    sanity_check(df)

    print(f"Done {lan} ]]]")

In [3]:
from mlds.data_loader import LANGUAGES


for lan in LANGUAGES:
    lan_entity = entity_manager.load_data(lan, reviewed=True, merged=True)
    merge_entity_intent(lan, lan_entity)

[[[ Start amh
lan_entity: 3200
lan_intent: 3200
3200
amh done, save file to data/output/amh.csv
3200
40
True
Done amh ]]]
[[[ Start ewe
lan_entity: 3200
lan_intent: 3200
3200
ewe done, save file to data/output/ewe.csv
3200
40
True
Done ewe ]]]
[[[ Start hau
lan_entity: 3200
lan_intent: 3200
3200
hau done, save file to data/output/hau.csv
3200
40
True
Done hau ]]]
[[[ Start ibo
lan_entity: 3200
lan_intent: 3200
3200
ibo done, save file to data/output/ibo.csv
3200
40
True
Done ibo ]]]
[[[ Start kin
lan_entity: 3200
lan_intent: 3200
3200
kin done, save file to data/output/kin.csv
3200
40
True
Done kin ]]]
[[[ Start lin
lan_entity: 3200
lan_intent: 3200
3200
lin done, save file to data/output/lin.csv
3200
40
True
Done lin ]]]
[[[ Start lug
lan_entity: 3200
lan_intent: 3200
3200
lug done, save file to data/output/lug.csv
3200
40
True
Done lug ]]]
[[[ Start orm
lan_entity: 3200
lan_intent: 3200
3200
orm done, save file to data/output/orm.csv
3200
40
True
Done orm ]]]
[[[ Start sna
lan_entity